# Propagação em espaço livre — Fundamentos FDTD

O **FDTD** (Finite-Difference Time-Domain) discretiza as equações de Maxwell no tempo e no espaço. Na **malha de Yee**, os campos E e H estão desenquadrados meio passo no espaço (e no tempo). A condição **CFL** garante estabilidade: `dt <= courant * min(dx,dy,dz)/c`.

Aqui: injectamos um pulso Gaussian no vácuo, gravamos Ey em dois pontos e **medimos a velocidade** por correlação cruzada. **Teoria: v = c**; comparamos com a simulação.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Project root (run notebook from project root, or we search upward)
ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "emsim").is_dir():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from emsim.fdtd.grid import YeeGrid
from emsim.fdtd.fields import update_E, update_H
from emsim.sources.gaussian_pulse import GaussianPulse
from emsim.constants import C0
from Tutorial.common.theory import measure_wave_speed

In [ ]:
# Grid: elongated in z for propagation; source and two recording points
grid = YeeGrid(
    x_range=(0, 5e-3),
    y_range=(0, 5e-3),
    z_range=(0, 60e-3),
    f0=10e9,
    resolution=40,
    courant=0.5,
    eps_r=1.0, mu_r=1.0, sigma=0.0,
)
z_src = 5
z1, z2 = 15, 35
distance = (z2 - z1) * grid.dz
source = GaussianPulse(f0=10e9, bandwidth=4e9)

Ey_at_z1, Ey_at_z2 = [], []
n_steps = 2000
mat = grid.materials
coeffs = grid.get_curl_coefficients()
inv_dx, inv_dy, inv_dz = coeffs["inv_dx"], coeffs["inv_dy"], coeffs["inv_dz"]

In [ ]:
for n in range(n_steps):
    update_H(grid.Ex, grid.Ey, grid.Ez, grid.Hx, grid.Hy, grid.Hz,
             mat.dt_over_mu, inv_dx, inv_dy, inv_dz)
    update_E(grid.Ex, grid.Ey, grid.Ez, grid.Hx, grid.Hy, grid.Hz,
             mat.Ca, mat.Cb, inv_dx, inv_dy, inv_dz)
    amplitude = source(n * grid.dt)
    amp = float(amplitude.numpy()) if hasattr(amplitude, "numpy") else float(amplitude)
    idx = tf.constant([[z_src, grid.Ny // 2, grid.Nx // 2]], dtype=tf.int32)
    new_val = grid.Ey[z_src, grid.Ny // 2, grid.Nx // 2].numpy() + amp
    grid.Ey.assign(tf.tensor_scatter_nd_update(
        grid.Ey.read_value(), idx, tf.constant([new_val], dtype=grid.Ey.dtype)
    ))
    Ey_at_z1.append(grid.Ey[z1, grid.Ny // 2, grid.Nx // 2].numpy())
    Ey_at_z2.append(grid.Ey[z2, grid.Ny // 2, grid.Nx // 2].numpy())

Ey_at_z1 = np.array(Ey_at_z1)
Ey_at_z2 = np.array(Ey_at_z2)
t = np.arange(n_steps) * grid.dt

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 4))
ax.plot(t * 1e9, Ey_at_z1, label=f"Ey em z1 (z={z1*grid.dz*1e3:.1f} mm)")
ax.plot(t * 1e9, Ey_at_z2, label=f"Ey em z2 (z={z2*grid.dz*1e3:.1f} mm)")
ax.set_xlabel("Tempo [ns]")
ax.set_ylabel("Ey [V/m]")
ax.legend()
ax.set_title("Campo Ey nos dois pontos de gravação; o pico em z2 atrasa em relação a z1")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
measured_speed = measure_wave_speed(Ey_at_z1, Ey_at_z2, distance, grid.dt)
error_pct = 100 * abs(measured_speed - C0) / C0

print("Comparação teoria vs simulação:")
print(f"  Teoria:     v = c = {C0:.3e} m/s")
print(f"  Simulação:  v_medido = {measured_speed:.3e} m/s")
print(f"  Erro:       {error_pct:.2f}%")

fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.bar(["Teoria (c)", "Simulação"], [C0, measured_speed], color=["C0", "C1"])
ax.set_ylabel("Velocidade [m/s]")
ax.set_title("Velocidade da onda: teoria vs simulação")
plt.tight_layout()
plt.show()